In [1]:
#!/usr/bin/env python
# coding: utf-8

"""
validate_segmentation.ipynb

Notebook de validação visual para segmentação de furos de um único drill.
Mostra espectrogramas com as janelas detectadas e os furos (jams) marcados.
"""

# =====================================================
# IMPORTS E CONFIGURAÇÃO DE AMBIENTE
# =====================================================
import os
import tempfile
import numpy as np
import librosa
import soundfile as sf

In [2]:
import os
import soundfile as sf
import numpy as np
import librosa
import pandas as pd

# =====================================================
# CONFIGURAÇÕES GERAIS
# =====================================================
STANDARDIZED_DIR = "data/standardized"
SEGMENTED_DIR = "data/segmented"
RAW_DIR = "data/raw"

ENERGY_THRESHOLD = 0.02
MIN_HOLE_DURATION = 0.5  # segundos
SILENCE_GAP = 0.2        # segundos de margem entre furos

def ensure_dir(path):
    os.makedirs(path, exist_ok=True)

# =====================================================
# FUNÇÕES AUXILIARES
# =====================================================
def load_jams(drill_folder):
    """Carrega furos com falha (travamento) listados em jams.txt"""
    if not drill_folder:
        return []
    jams_path = os.path.join(drill_folder, "jams.txt")
    if os.path.exists(jams_path):
        try:
            with open(jams_path, "r", encoding="utf-8", errors="ignore") as f:
                content = f.read().strip()
            if content:
                return [int(x.strip()) for x in content.split(",") if x.strip().isdigit()]
        except Exception as e:
            print(f"⚠️ Erro ao ler {jams_path}: {e}")
    return []

def segment_holes(y, sr, energy_threshold=ENERGY_THRESHOLD, min_hole_duration=MIN_HOLE_DURATION):
    """
    Detecta furos (intervalos de baixa energia entre cortes).
    Retorna uma lista [(inicio, fim)] em amostras.
    """
    hop_length = 512
    frame_length = 1024

    rms = librosa.feature.rms(y=y, frame_length=frame_length, hop_length=hop_length)[0]
    max_rms = np.max(rms) if np.max(rms) > 0 else 1.0
    rms_norm = rms / max_rms

    # Furo = separação entre blocos de energia alta
    mask = rms_norm > energy_threshold
    holes = []
    start = None

    for i, active in enumerate(mask):
        if active and start is None:
            start = i
        elif not active and start is not None:
            end = i
            start_sample = int(start * hop_length)
            end_sample = int(end * hop_length)
            if (end_sample - start_sample) / sr >= min_hole_duration:
                holes.append((start_sample, end_sample))
            start = None

    if start is not None:
        holes.append((int(start * hop_length), len(y)))

    return holes

# =====================================================
# PROCESSAMENTO PRINCIPAL
# =====================================================
def process_drill_folder(drill_id, jams):
    src_dir = os.path.join(STANDARDIZED_DIR, drill_id)
    dst_dir = os.path.join(SEGMENTED_DIR, drill_id)
    ensure_dir(dst_dir)

    metadata_list = []

    for file in sorted(os.listdir(src_dir)):
        if not file.lower().endswith(".wav"):
            continue

        filepath = os.path.join(src_dir, file)

        try:
            y, sr = sf.read(filepath)
            if len(y.shape) > 1:
                y = np.mean(y, axis=1)
        except Exception as e:
            print(f"⚠️ Erro ao ler {file}: {e}")
            continue

        holes = segment_holes(y, sr)
        print(f"\n🎧 {file} — {len(holes)} furos detectados")

        # Extrai info do nome
        parts = file.replace(".wav", "").split("_")
        if len(parts) < 4:
            print(f"⚠️ Nome inesperado: {file}")
            continue

        drill_id = parts[0].replace("drill", "")
        mic_type = parts[1]
        mic_id = parts[2].replace("Tr", "")
        position = parts[3]

        for idx, (start, end) in enumerate(holes, 1):
            hole_audio = y[start:end]
            duration = (end - start) / sr
            jam_tag = "_jam" if idx in jams else ""
            out_name = f"drill{drill_id}_hole{idx:02d}_{mic_type}_{mic_id}_{position}{jam_tag}.wav"
            out_path = os.path.join(dst_dir, out_name)

            # adiciona pequena margem antes e depois
            pad = int(SILENCE_GAP * sr)
            start_padded = max(0, start - pad)
            end_padded = min(len(y), end + pad)
            segment = y[start_padded:end_padded]

            sf.write(out_path, segment, sr)
            print(f"   💾 {out_name} ({duration:.2f}s)")

            metadata_list.append({
                "drill_id": drill_id,
                "hole_idx": idx,
                "mic_type": mic_type,
                "mic_id": mic_id,
                "position": position,
                "jam": idx in jams,
                "sr": sr,
                "duration_s": duration,
                "filepath": out_path
            })

    return metadata_list


# =====================================================
# MAIN
# =====================================================
def main():
    ensure_dir(SEGMENTED_DIR)
    all_metadata = []

    for drill_folder in sorted(os.listdir(STANDARDIZED_DIR)):
        full_path = os.path.join(STANDARDIZED_DIR, drill_folder)
        if not os.path.isdir(full_path):
            continue

        print(f"\n📦 Processando {drill_folder}")
        raw_match = [
            r for r in os.listdir(RAW_DIR)
            if drill_folder in r or drill_folder.replace('drill_', '') in r
        ]
        raw_path = os.path.join(RAW_DIR, raw_match[0]) if raw_match else None
        jams = load_jams(raw_path)
        print(f"   → Furos com jam: {jams if jams else 'nenhum'}")

        metadata = process_drill_folder(drill_folder, jams)
        all_metadata.extend(metadata)

    # salva CSV consolidado
    df = pd.DataFrame(all_metadata)
    csv_path = os.path.join(SEGMENTED_DIR, "metadata_segmented.csv")
    df.to_csv(csv_path, index=False)
    print(f"\n📊 Metadata de furos salva em: {csv_path}")


if __name__ == "__main__":
    main()



📦 Processando 01
   → Furos com jam: [39, 10, 3]

🎧 01_common_1_ext.wav — 17 furos detectados
   💾 drill01_hole01_common_1_ext.wav (0.66s)
   💾 drill01_hole02_common_1_ext.wav (0.82s)
   💾 drill01_hole03_common_1_ext_jam.wav (1.07s)
   💾 drill01_hole04_common_1_ext.wav (2.19s)
   💾 drill01_hole05_common_1_ext.wav (0.67s)
   💾 drill01_hole06_common_1_ext.wav (1.81s)
   💾 drill01_hole07_common_1_ext.wav (1.49s)
   💾 drill01_hole08_common_1_ext.wav (3.98s)
   💾 drill01_hole09_common_1_ext.wav (0.58s)
   💾 drill01_hole10_common_1_ext_jam.wav (1.86s)
   💾 drill01_hole11_common_1_ext.wav (512.86s)
   💾 drill01_hole12_common_1_ext.wav (96.30s)
   💾 drill01_hole13_common_1_ext.wav (73.42s)
   💾 drill01_hole14_common_1_ext.wav (51.80s)
   💾 drill01_hole15_common_1_ext.wav (0.81s)
   💾 drill01_hole16_common_1_ext.wav (0.96s)
   💾 drill01_hole17_common_1_ext.wav (0.49s)

🎧 01_common_2_ext.wav — 14 furos detectados
   💾 drill01_hole01_common_2_ext.wav (1.48s)
   💾 drill01_hole02_common_2_ext.wav 

In [ ]:
# coloque sem filepath (nova célula)
y, sr = librosa.load(<caminho_arquivo_wav_de_teste>, sr=None, mono=True)
hop_length = 512; frame_length = 1024
rms = librosa.feature.rms(y=y, frame_length=frame_length, hop_length=hop_length)[0]
print("rms.min, rms.max, rms.mean:", rms.min(), rms.max(), rms.mean())
print("rms_norm sample:", (rms/ (np.max(rms) if np.max(rms)>0 else 1.0))[:20])
mask = (rms / (np.max(rms) if np.max(rms)>0 else 1.0)) < ENERGY_THRESHOLD
print("mask true frames count:", mask.sum())
holes = segment_holes(y, sr)
print("holes detected (s):", [(s/sr, e/sr) for s,e in holes])